In [1]:
from pyCHX.chx_packages import *
from tqdm import tqdm
from zoneinfo import ZoneInfo
import sys
sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/database_processing/") #is there a way to include all subfolders?
sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/standard_functions/")
sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/AutoRun_functions/")
from database_processing import *
from standard_functions import *
from AutoRun_functions import *

%run /nsls2/data/chx/legacy/analysis/2022_3/lwiegart/development/chx_analysis_setup.ipynb

OBJECT CACHE: Will use up to 121_499_065_344 bytes (15% of total physical RAM)
/nsls2/conda/envs/2024-1.0-py311-tiled-rc1/lib/python3.11/site-packages/databroker/v1.py:72: UserWarning: In databroker 2.x, there are separate notions of 'server' and 'client', and register_handler(...) has no effect on the client. Likely this is being done for you on the server side, so you should not worry about this message unless you encounter trouble loading large array data.
  warnings.warn(


running on: jupyter_hub   environment: standard
setting '_base_path_' as /nsls2/data/chx/legacy/analysis/
ran "%run -i /nsls2/data/chx/legacy/analysis/2022_3/lwiegart/development/roi_nr_2019_3_0_1.py" to fix ROI numbering for PHI-sliced data sets. Should get fixed in pyCHX...
ran "%run -i /nsls2/data/chx/legacy/analysis/2023_2/lwiegart/pyCHX_June2023/chx_outlier_detection.py": this should become part of pyCHX.
ran "%run -i /nsls2/data/chx/legacy/analysis/2023_2/lwiegart/pyCHX_June2023/fix_get_sid_filenames.py": this should be fixed and re-deployed in pyCHX.

environment dependent settings and patches:
using "%matplotlib inline" for plotting
ran "%run -i /nsls2/data/chx/legacy/analysis/2022_3/lwiegart/development/polygonmask_fix.py".
ran "%run -i /nsls2/data/chx/legacy/analysis/2023_2/lwiegart/pyCHX_June2023/chx_compress.py": temporary fix for depreciated np.float.
ran "%run -i /nsls2/data/chx/legacy/analysis/2023_2/lwiegart/pyCHX_June2023/chx_correlationc.py": temporary fix for depreci

### Define list of datasets to (re-)run
Possible ways to define a list of datasets:
1) list of uids (complete or partial) or list of scan_ids, IF scan_ids CAN provide additional information 'user' and 'cycle' to increase the change of finding a single result; can mix scan_ids and uids:

scans = ['a2342-ae23421-2112',143231] -> this will just re-run the notebook specified in the respective metadata  
user='DonaldDuck'  
cycle='2050_2'  

2) change the way the analysis is being run: change the Q-Phi-mask to another standard mask (check_roi_masknames(_base_path_+'masks') returns a list of currently available standard masks) or provide [start,stop] frames to calculate g_2 over a limited range  [only effective for processing notebooks created after XYZ]: 

scans = ['a2342-ae23421-2112',[143231,[20,230],'phi_16x22_5deg']]  
user='DonaldDuck'  
cycle='2050_2'  

#### re-run data processing without modification

In [2]:
if False:
    scans = ['1c18fa25',138836]
    user = 'commissioning'
    cycle = '2023_3'
    
    ##############################################################################################################
    if user: user_=user
    if cycle: cycle_=cycle
    [s,uids]=get_uid_list(scans,user=user_,cycle=cycle_)

#### re-run data processing with modifications: manual list

In [3]:
print('available standard Q-Phi-Masks: ',check_roi_masknames(_base_path_+'masks'))

available standard Q-Phi-Masks:  ['phi_12x30r45deg', 'wide', 'phi_16x22_5deg', 'phi_12x_30deg', 'phi_4x_20deg', 'phi_12x_30deg_flow', 'phi_12x_15deg_flow', 'norm']


In [4]:
if True:
    scans = [['1c18fa25','phi_12x_30deg'],[138836,[5,150],'phi_4x_20deg'],'077ddcd1']
    user = 'commissioning'
    cycle = '2023_3'
    
    ##############################################################################################################
    if user: user_=user
    if cycle: cycle_=cycle
    uids=[]
    for s in scans:
        if type(s) == list:
            [a,tmp]=get_uid_list([s[0]],user=user_,cycle=cycle_)
            s[0]=tmp[0]
            uids.append(s)
        else:
            [a,tmp]=get_uid_list([s],user=user_,cycle=cycle_)
            uids.append(tmp[0])

#### re-run data processing with modifications: change Q-Phi-mask for batch processing

In [5]:
if False:
    scans = np.arange(138825,138830,dtype=int)
    mask = 'phi_4x_20deg'
    user = 'commissioning'
    cycle = '2023_3'
    
    ##############################################################################################################
    if user: user_=user
    if cycle: cycle_=cycle
    uids=[]
    for s in scans:
        [a,tmp]=get_uid_list([int(s)],user=user_,cycle=cycle_)
        uids.append([tmp[0],mask])

### Check if the requested Processing is possible:

In [6]:
for u in uids:
    d=do_analysis_setup(u,_base_path_,verbose=False)
    print('\n',d)
    image_range=None
    if 'rstart' in list(d.keys()):
        image_range=[d['rstart'],d['rstop']]
    check_auto_processing_possible(d['uid'],_base_path_,image_range=image_range,verbose=True)


 {'uid': '1c18fa25-98cf-4de7-92f6-fd03e43c69e7', 'analysis_type_auto': False, 'qphi_analysis': True, 'phi_Q_mask': 'phi_12x_30deg'}
summary for auto-processing check for scan_id: 138837 uid 1c18fa25-98cf-4de7-92f6-fd03e43c69e7: automated processing should be possible!

 {'uid': '2b60052a-6cca-4613-9781-edb22bd15248', 'rstart': 5, 'rstop': 150, 'analysis_type_auto': False, 'qphi_analysis': True, 'phi_Q_mask': 'phi_4x_20deg'}
summary for auto-processing check for scan_id: 138836 uid 2b60052a-6cca-4613-9781-edb22bd15248: automated processing should be possible!

 {'uid': '077ddcd1-600a-4cee-a900-6108462aca11'}
summary for auto-processing check for scan_id: 138831 uid 077ddcd1-600a-4cee-a900-6108462aca11: automated processing should be possible!


### Batch Process datasets

In [7]:
track_progress = True # IF True: track progress via entires in json dictionary [currently only implemented in new XPCS_SAXS_auto processing notebook]

t=str(datetime.datetime.now(ZoneInfo("America/New_York")))
txt_filename = 'BatchProcessing_'+t.split()[0]+'_%s_%s_%s.txt'%(t.split()[1].split(':')[0],t.split()[1].split(':')[1],t.split()[1].split(':')[2].split('.')[0])
a=%pwd
txt_path='%s/BatchProcessing/'%a
if not os.path.exists(txt_path):
    os.makedirs(txt_path)

if track_progress:
    processing_overview_dict_file = 'default' # IF default: directory_this_notebook_is_running/processing_overview_dict.json
    if processing_overview_dict_file == 'default':
        a=%pwd
        processing_overview_dict_file=a+'/processing_overview_dict.json'

for uu,u in enumerate(uids):
    clear_output();insert_dict=None
    try:
        if type(u)==str: md=get_meta_data(u); scan_id = md['scan_id'];uid=u
        elif type(u)==list: md=get_meta_data(u[0]); scan_id = md['scan_id'];uid=u[0]
        if track_progress:
            progress_dict_file = get_progress_dict_filename(uid,md['cycle'],md['user'],_base_path_)
            process_id = get_process_id(uid,md['cycle'],md['user'],_base_path_,verbose=False) # -> need to pass this to notebook
            manage_processing_overview(processing_overview_dict_file,process_id,action='start_processing',progress_dict_file=progress_dict_file,machine=machine,verbose=True)
            insert_dict={'process_id':process_id}
        print("Batch processing %s/%s -> analyzing scan_id: %s with processing instructions %s:"%(uu+1,len(uids),scan_id,u))
        chx_analysis_data( u , _base_path_, alternate_directory=None, insert_dict=insert_dict)
        status='success'
        if track_progress:
            manage_processing_overview(processing_overview_dict_file,process_id,action='finished_processing',machine=None,verbose=True)
    except:
        status='failed'
        if track_progress:
            manage_processing_overview(processing_overview_dict_file,process_id,action='failed_processing',machine=None,verbose=True)
    append_txtfile(txt_path+txt_filename,['scan_id: %s  status: %s  instructions: %s'%(scan_id,status,u)],delimiter=',')
print('Summary of Batch Processing has been saved in %s'%txt_filename)    

updating processing_overview_dict: process_id = 077ddcd1-600a-4cee-a900-6108462aca11_5 -> start
Batch processing 3/3 -> analyzing scan_id: 138831 with processing instructions 077ddcd1-600a-4cee-a900-6108462aca11:


Executing:   0%|          | 0/162 [00:00<?, ?cell/s]

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


updating processing_overview_dict: process_id = 077ddcd1-600a-4cee-a900-6108462aca11_5 -> completed
Summary of Batch Processing has been saved in BatchProcessing_2024-02-27_16_33_16.txt
